# Federated PCA Demonstartion on 1000 Genomes Data

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:

import sys
sys.path.insert(1, '/home/genxadmin/uk-biobank/src/preprocess')

import subprocess

import pandas as pd
import matplotlib.pyplot as plt

from preprocess.federated_pca import FederatedPCASimulationRunner

### [ 0 ] Create Data Folders

In [ ]:
SPLIT_DIRECTORY = '/mnt/genx-bio-share/TG/data/chip/superpop_split'
federated_pca_runner = FederatedPCASimulationRunner(
    source_folder=f'{SPLIT_DIRECTORY}/genotypes',
    result_folder=f'{SPLIT_DIRECTORY}/federated_pca',
    variant_ids_file=f'{SPLIT_DIRECTORY}/genotypes/ALL.prune.in',
    n_components=20,
    method='P-STACK',
    nodes=['AFR', 'AMR', 'EAS', 'SAS', 'EUR']
)

# List of clients nodes
print(federated_pca_runner.nodes)

### [ 1 ] Variants Pruning (Simulation)

In [ ]:
from preprocess.pruning import PlinkPruningRunner

PlinkPruningRunner(
    source_directory=f'{SPLIT_DIRECTORY}/genotypes',
    nodes=['AFR', 'AMR', 'EAS', 'SAS', 'EUR'],
    result_filepath=f'{SPLIT_DIRECTORY}/genotypes/ALL.prune.in',
    node_filename_template='%s_filtered'
).run(window_size=1000, step=50, threshold=0.001)

In [ ]:
variant_ids_file = f'{SPLIT_DIRECTORY}/genotypes/ALL.prune.in'

print('The number of ramaining variants')
!wc -l $variant_ids_file

### [ 2 ] Federated PCA

In [ ]:
federated_pca_runner.run()

### [ 3 ] Plot Projections Results for All Nodes

In [ ]:
colors = {
    'AFR': 'black',
    'SAS': 'blue',
    'EUR': 'red',
    'AMR': 'green',
    'EAS': 'grey'
}

for node in federated_pca_runner.nodes:
    evectors = pd.read_csv(f'{SPLIT_DIRECTORY}/federated_pca/{node}/fold_9_train_projections.csv.eigenvec.sscore', sep='\t')
    evectors = evectors.drop(evectors.columns[1:3], axis=1).rename(columns={'#IID': 'IID'}).set_index('IID')
    plt.scatter(evectors.iloc[:, [1]], evectors.iloc[:, [2]], c=colors[node])

plt.show()

In [ ]:
evectors = pd.read_csv(f'{SPLIT_DIRECTORY}/federated_pca/ALL/fold_3_train_projections.csv.eigenvec.sscore', sep='\t')
evectors = evectors.drop(evectors.columns[1:3], axis=1).rename(columns={'#IID': 'IID'}).set_index('IID')
samples = pd.read_csv('/mnt/genx-bio-share/TG/data/chip/igsr_samples.tsv', sep='\t', header=0)
codes = pd.merge(evectors, samples, left_on='IID', right_on='Sample name')['Superpopulation code'].to_numpy()

for node in federated_pca_runner.nodes:
    selector = (codes == node)
    plt.scatter(evectors.iloc[selector, [7]], evectors.iloc[selector, [8]], c=colors[node])

plt.show()

In [ ]:
evectors = pd.read_csv(f'{SPLIT_DIRECTORY}/federated_pca/ALL/fold_9_train_projections.csv.eigenvec.sscore', sep='\t')
evectors = evectors.drop(evectors.columns[1:3], axis=1).rename(columns={'#IID': 'IID'}).set_index('IID')
samples = pd.read_csv('/mnt/genx-bio-share/TG/data/chip/igsr_samples.tsv', sep='\t', header=0)
codes = pd.merge(evectors, samples, left_on='IID', right_on='Sample name')['Superpopulation code'].to_numpy()

for node in federated_pca_runner.nodes:
    selector = (codes == node)
    plt.scatter(evectors.iloc[selector, [7]], evectors.iloc[selector, [8]], c=colors[node])

plt.show()

In [ ]:
evectors.head()

### [ * ] Centralized PCA with Plink

In [ ]:
subprocess.run(['plink2',
    '--pfile', f'{SPLIT_DIRECTORY}/genotypes/ALL/fold_0_train',
    '--extract', f'{SPLIT_DIRECTORY}/genotypes/ALL.prune.in',
    '--freq', 'counts',
    '--out',  f'{SPLIT_DIRECTORY}/pca/ALL/fold_0_train_projections',
    '--pca', 'allele-wts', '20'
])

for part in ['train', 'val', 'test']:
    subprocess.run(['plink2',
        '--pfile', f'{SPLIT_DIRECTORY}/genotypes/ALL/fold_0_{part}',
        '--extract', f'{SPLIT_DIRECTORY}/genotypes/ALL.prune.in',
        '--read-freq', f'{SPLIT_DIRECTORY}/pca/ALL/fold_0_train_projections.acount',
        '--score', f'{SPLIT_DIRECTORY}/pca/ALL/fold_0_train_projections.eigenvec.allele',
            '2', '5', 'header-read', 'no-mean-imputation', 'variance-standardize',
        '--score-col-nums', '6-25',
        '--out', f'{SPLIT_DIRECTORY}/pca/ALL/fold_0_{part}_projections.csv.eigenvec'
    ])

In [ ]:
samples = pd.read_csv('/mnt/genx-bio-share/TG/data/chip/igsr_samples.tsv', sep='\t', header=0)

input_sscore_file = f'{SPLIT_DIRECTORY}/pca/ALL/fold_9_train_projections.csv.eigenvec.sscore'
sscore = pd.read_csv(input_sscore_file, sep='\t', header=0)
codes = pd.merge(sscore, samples, left_on='#IID', right_on='Sample name')['Superpopulation code'].to_numpy()

evectors = sscore[sscore.columns[3:]]
for node in federated_pca_runner.nodes:
    selector = (codes == node)
    plt.scatter(-evectors.iloc[selector, [0]], evectors.iloc[selector, [1]], c=colors[node])

### [ * ] All in Once

In [ ]:
import os
import time


n_components = 10
portions = [0.0005, 0.001, 0.002, 0.003, 0.004, 0.005, 0.01]
ns_variants = []
elapsed_times = []
communication_costs = [] # FIXME: how to measure it?

for iteration, portion in enumerate(portions):
    print(f'MEASUREMENT {iteration + 1} OF {len(portions)}')

    federated_pca.prune(portion=portion)
    variant_ids_file = federated_pca.VARIANT_IDS_FOLDER + '/pruned.ids'
    n_variants = pd.read_csv(variant_ids_file, sep='\t', header=None).shape[0]
    ns_variants.append(n_variants)

    total_time = 0
    start = time.time()

    federated_pca.compute_allele_frequencies(federated_pca.ALL, variant_ids_file=variant_ids_file)
    allele_frequencies_file = federated_pca.PCA_FOLDER + '/ALL.acount'
    for node in federated_pca.NODES:
        federated_pca.client(
            node=node,
            variant_ids_file=variant_ids_file,
            allele_frequencies_file=allele_frequencies_file
        )

    evectors, evalues = federated_pca.server(federated_pca.NODES, n_components=n_components)
    federated_pca.create_plink_eigenvec_allele_file(evectors, n_components=n_components)

    plink_allele_projector = federated_pca.PCA_FOLDER + '/federated.eigenvec.allele'
    for node in federated_pca.NODES:
        federated_pca.client_projection(
            node=node,
            variant_ids_file=variant_ids_file,
            allele_frequencies_file=allele_frequencies_file,
            plink_allele_projector=plink_allele_projector,
            n_components=n_components
        )

    end = time.time()
    elapsed = end - start
    elapsed_times.append(elapsed)

    # Communication costs (Mb)
    cost = 0

    # [1] Alleles counts aggregation (approximation)
    acount_file = os.path.join(federated_pca.PCA_FOLDER, federated_pca.ALL + '.acount')
    cost += 6 * os.path.getsize(acount_file) / 1024 / 1024
    
    # [2] Local PCA results
    for node in federated_pca.NODES:
        eigenval_file = os.path.join(federated_pca.PCA_FOLDER, node + '.eigenval')
        eigenvec_allele_file = os.path.join(federated_pca.PCA_FOLDER, node + '.eigenvec.allele')

        cost += os.path.getsize(eigenval_file) / 1024 / 1024
        cost += os.path.getsize(eigenvec_allele_file) / 1024 / 1024

    # [3] Result PCA projection file to clients
    federated_eigenvec_allele_file = os.path.join(federated_pca.PCA_FOLDER, 'federated.eigenvec.allele')
    cost += os.path.getsize(federated_eigenvec_allele_file) / 1024 / 1024
    
    communication_costs.append(cost)

In [ ]:
plt.plot(ns_variants, elapsed_times)
plt.xlabel('n_variants')
plt.ylabel('time [sec]')

In [ ]:
plt.plot(ns_variants, communication_costs)
plt.xlabel('n_variants')
plt.ylabel('communication_costs [Mb]')

### ...

In [ ]:
import pandas as pd
import numpy as np

file = '/mnt/genx-bio-share/TG/data/chip/superpop_split/federated_pca/ALL/fold_9_train.eigenvec.allele'
fed = pd.read_csv(file, sep='\t', header=0)

np.linalg.norm(fed['PC1'])


In [ ]:
np.linalg.norm(fed['PC11'])

In [ ]:
file = '/mnt/genx-bio-share/TG/data/chip/superpop_split/pca/ALL/fold_9_train_projections.eigenvec.allele'
cnt = pd.read_csv(file, sep='\t', header=0)

np.linalg.norm(cnt['PC1'])